# Regressió logística

**Optativa d'Aprenentatge automàtic · DAM/DAW 2n**

## 1. Una manera diferent de decidir

Els models que heu vist fins ara decidien de dues maneres. El k-NN preguntava
**«a qui t'assembles»**: mirava els veïns més propers i copiava la seva classe. Els
arbres i els boscos preguntaven **«quin valor té aquesta columna»**: feien preguntes de
sí/no en cadena fins arribar a una resposta.

Avui fem una cosa diferent: **traçar una ratlla i mirar de quin costat caus**. Sona
senzill, i de fet ho és, però aquesta idea és la base de bona part del *machine
learning* modern: xarxes neuronals, SVM... totes comencen aquí.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

iris = load_iris(as_frame=True)
dades = iris.frame.copy()
dades["especie"] = iris.target_names[iris.target]
dades = dades.rename(columns={
    "sepal length (cm)": "sepal_llarg",
    "sepal width (cm)": "sepal_ample",
    "petal length (cm)": "petal_llarg",
    "petal width (cm)": "petal_ample",
})

colors3 = {"setosa": "tab:blue", "versicolor": "tab:orange", "virginica": "tab:green"}

dades.head()

## 2. De la recta a la probabilitat

### 2.1 La recta que ja coneixeu

Ens quedem amb dues espècies que costen de separar: **versicolor** i **virginica**
(la tercera, *setosa*, es distingeix tan bé que no fa cap gràcia). I ens quedem amb
dues columnes: la llargada i l'amplada del pètal.

Ja sabeu dibuixar una recta: $y = mx + n$. Escrita d'una altra manera, amb les dues
columnes com a variables $x_1$ (llargada) i $x_2$ (amplada):

$$w_1 x_1 + w_2 x_2 + b = 0$$

Els números $w_1$, $w_2$ i $b$ es diuen **pesos** i **biaix**. Trien la recta: canvieu-los
i la recta es mou. El que ens interessa no és la recta en si, sinó el **signe** del que
hi ha a l'esquerra:

- Si $w_1 x_1 + w_2 x_2 + b > 0$, ets a un costat.
- Si $w_1 x_1 + w_2 x_2 + b < 0$, ets a l'altre.

Provem-ho amb uns pesos triats a ull, $w_1 = 1$, $w_2 = 1$, $b = -6{,}5$, és a dir, la
recta $x_1 + x_2 = 6{,}5$.

In [ ]:
bi = dades[dades["especie"].isin(["versicolor", "virginica"])].copy()
bi["y"] = (bi["especie"] == "virginica").astype(int)  # 0 = versicolor, 1 = virginica
colors_bi = {"versicolor": "tab:orange", "virginica": "tab:green"}

w1_ma, w2_ma, b_ma = 1.0, 1.0, -6.5

plt.figure(figsize=(8, 5))
for especie, grup in bi.groupby("especie"):
    plt.scatter(grup["petal_llarg"], grup["petal_ample"], label=especie,
                color=colors_bi[especie], s=40, alpha=0.8)

x1_vals = np.linspace(bi["petal_llarg"].min() - 0.3, bi["petal_llarg"].max() + 0.3, 100)
x2_vals = -(w1_ma * x1_vals + b_ma) / w2_ma
plt.plot(x1_vals, x2_vals, "k--", label=f"$x_1 + x_2 = {-b_ma:.1f}$")

plt.xlabel("Llargada del pètal (cm)")
plt.ylabel("Amplada del pètal (cm)")
plt.title("Una recta triada a ull")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

z = w1_ma * bi["petal_llarg"] + w2_ma * bi["petal_ample"] + b_ma
pred = (z > 0).map({True: "virginica", False: "versicolor"})
precisio = (pred == bi["especie"]).mean()
print(f"Precisió de la recta triada a ull: {precisio:.1%}  ({(pred == bi['especie']).sum()} de {len(bi)})")

### 2.2 El problema del signe, i la sigmoide

Un 95 % amb una recta triada a ull. Però fixeu-vos en què dona el signe: només un **sí**
o un **no**. Una flor que cau just tocant la ratlla compta exactament igual que una que
cau ben endins d'un costat, i això no té sentit: no n'esteu igual de segurs.

Voldríem un número que digués **com de segurs n'estem**, no només de quin costat caiem.
Per això es fa servir la **funció sigmoide**:

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

on $z = w_1 x_1 + w_2 x_2 + b$, el mateix número de la recta que ja teníeu.

No cal derivar-la per entendre-la: **mireu-ne el gràfic**. Tres propietats hi salten
a la vista:

- Sempre val **entre 0 i 1**: es pot llegir com una probabilitat.
- Quan $z = 0$ (ets just sobre la recta), val **0,5**: no en sap res, tira moneda.
- Quan $z$ és molt gran o molt petit, la corba **s'aplana** a prop de 1 o de 0: com
  més endins ets d'un costat, més segur n'està, però amb rendiments decreixents.

El signe només us deia el costat. La sigmoide converteix «quant endins ets» en una
probabilitat.

In [ ]:
def sigmoide(z):
    return 1 / (1 + np.exp(-z))

z_vals = np.linspace(-10, 10, 200)

plt.figure(figsize=(8, 4.5))
plt.plot(z_vals, sigmoide(z_vals), color="tab:red", linewidth=2)
plt.axhline(0.5, color="gray", linestyle=":", linewidth=1)
plt.axvline(0, color="gray", linestyle=":", linewidth=1)
plt.scatter([0], [0.5], color="black", zorder=5)
plt.xlabel("$z = w_1 x_1 + w_2 x_2 + b$")
plt.ylabel(r"$\sigma(z)$")
plt.title("La funció sigmoide")
plt.grid(alpha=0.3)
plt.show()

### 2.3 Fem-ho concret, amb tres flors

Agafem tres flors reals del conjunt de dades i els pesos $w_1=1$, $w_2=1$, $b=-6{,}5$
d'abans:

- Una **clarament versicolor**: pètal de 3,0 × 1,1 cm.
- Una **clarament virginica**: pètal de 6,9 × 2,3 cm.
- Una de **dubtosa**: pètal de 4,8 × 1,8 cm. Aquesta mesura exacta apareix al conjunt
  de dades real tant en flors etiquetades com *versicolor* com en flors etiquetades
  com *virginica*: fins i tot els experts que van mesurar-les es van trobar dues flors
  gairebé indistingibles.

In [ ]:
flors = pd.DataFrame({
    "flor": ["clarament versicolor", "dubtosa", "clarament virginica"],
    "petal_llarg": [3.0, 4.8, 6.9],
    "petal_ample": [1.1, 1.8, 2.3],
})
flors["z"] = w1_ma * flors["petal_llarg"] + w2_ma * flors["petal_ample"] + b_ma
flors["sigma(z)"] = sigmoide(flors["z"])
flors

## 3. Com s'aprenen els pesos, sense derivades

Els pesos $w_1=1$, $w_2=1$, $b=-6{,}5$ els hem triat a ull mirant el gràfic. Amb dues
columnes encara es pot fer. Amb cinquanta columnes, no.

Cal una manera perquè la màquina trobi els pesos sola. Per fer-ho li cal un
**termòmetre**: un número que digui com de malament ho està fent, perquè després pugui
buscar els pesos que el facin més petit. Aquest número es diu **funció de pèrdua**.

Per a classificació es fa servir la **log-loss** (o entropia creuada):

$$\text{pèrdua} = -\frac{1}{n}\sum_{i=1}^{n} \Big[ y_i \log(\sigma(z_i)) + (1-y_i)\log(1-\sigma(z_i)) \Big]$$

on $y_i$ és 0 o 1 (l'espècie real) i $\sigma(z_i)$ és la probabilitat que ha donat el
model. En paraules:

- Si la flor és virginica ($y_i=1$) i el model li dona una probabilitat alta, el primer
  terme aporta poc a la pèrdua. Si li dona una probabilitat baixa, el terme
  $\log(\sigma(z_i))$ es fa molt negatiu i **la pèrdua es dispara**.
- Igual a l'inrevés per a versicolor.

La idea de fons: **castiga molt estar molt segur i equivocar-te**. Una probabilitat de
0,51 quan t'equivoques és un error petit. Una probabilitat de 0,99 quan t'equivoques és
un error molt car.

In [ ]:
def perdua(w, b, X, y):
    w1, w2 = w
    z = w1 * X[:, 0] + w2 * X[:, 1] + b
    p = sigmoide(z)
    eps = 1e-12
    p = np.clip(p, eps, 1 - eps)  # evita log(0)
    return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))

X = bi[["petal_llarg", "petal_ample"]].values
y = bi["y"].values

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

# entrenem ja scikit-learn per tenir un biaix (b) raonable amb el qual moure's;
# el motiu del seu valor el veurem a la secció 4
model = LogisticRegression(random_state=42)
model.fit(Xtr, ytr)
b_bo = model.intercept_[0]

print(f"Biaix fixat per a la graella: b = {b_bo:.4f}\n")
for w1, w2 in [(0, 0), (1, 1), (2, 2), (2.5, 2.2)]:
    print(f"w1={w1:>4}, w2={w2:>4}  ->  pèrdua = {perdua((w1, w2), b_bo, Xtr, ytr):.4f}")

Com més a prop dels pesos «bons» ($w_1\approx2{,}5$, $w_2\approx2{,}2$), més baixa la
pèrdua. Enlloc de continuar provant combinacions a mà, **recorrem sistemàticament totes
les combinacions** de $w_1$ i $w_2$ en una graella (deixant $b$ fixat, perquè amb dues
incògnites ho podem dibuixar en un pla) i calculem la pèrdua a cadascuna. El resultat
és un **paisatge**: un mapa amb valls i pujades, on volem trobar el punt més baix.

In [ ]:
w1_vals = np.linspace(0, 5, 60)
w2_vals = np.linspace(0, 5, 60)
W1, W2 = np.meshgrid(w1_vals, w2_vals)

L = np.zeros_like(W1)
for i in range(W1.shape[0]):
    for j in range(W1.shape[1]):
        L[i, j] = perdua((W1[i, j], W2[i, j]), b_bo, Xtr, ytr)

w1_sk, w2_sk = model.coef_[0]

plt.figure(figsize=(7, 6))
cs = plt.contourf(W1, W2, L, levels=25, cmap="viridis")
plt.colorbar(cs, label="Pèrdua (log-loss)")
plt.scatter([w1_sk], [w2_sk], color="red", s=140, marker="*", zorder=5,
            label="Pesos trobats per scikit-learn")
plt.xlabel("$w_1$ (pes de la llargada del pètal)")
plt.ylabel("$w_2$ (pes de l'amplada del pètal)")
plt.title("El paisatge de la pèrdua")
plt.legend()
plt.show()

perdua_sk = perdua((w1_sk, w2_sk), b_bo, Xtr, ytr)
pct_pitjor = (L > perdua_sk).mean()
print(f"Pèrdua al punt de scikit-learn: {perdua_sk:.4f}")
print(f"El {pct_pitjor:.0%} de tota la graella té una pèrdua pitjor (més alta) que aquest punt.")

El punt vermell cau ben endins de la zona fosca: el fons de la vall. No és exactament
el punt més baix de tota la graella (la nostra pèrdua, simplificada, encara pot baixar
una mica més cap a pesos molt grans; scikit-learn hi afegeix un petit ajust, la
*regularització*, perquè això no passi) però la idea central ja hi és.

I amb això ja podem dir, sense cap fórmula de derivades, què fa realment
l'entrenament: **baixar per aquest paisatge fins al fons, a passes petites**, provant
en quina direcció la pèrdua baixa més i movent-s'hi. Això es diu **descens de
gradient**. Nosaltres l'hem simulat provant tota la graella; un ordinador, amb
centenars de pesos, no pot provar-ho tot i baixa el pendent pas a pas. El detall de com
calcula el pendent queda per a qui vulgui estirar-ne el fil.

## 4. Compara amb scikit-learn

Ja tenim `model`, entrenat unes cel·les més amunt. Mirem exactament què ha après.

In [ ]:
print(f"w1 = {model.coef_[0][0]:.4f}")
print(f"w2 = {model.coef_[0][1]:.4f}")
print(f"b  = {model.intercept_[0]:.4f}")
print(f"\nPrecisió a l'examen: {model.score(Xte, yte):.1%} "
      f"({int(round(model.score(Xte, yte) * len(Xte)))} de {len(Xte)})")

w1_final, w2_final = model.coef_[0]
b_final = model.intercept_[0]

plt.figure(figsize=(8, 5))
for especie, grup in bi.groupby("especie"):
    plt.scatter(grup["petal_llarg"], grup["petal_ample"], label=especie,
                color=colors_bi[especie], s=40, alpha=0.8)

x1_vals = np.linspace(bi["petal_llarg"].min() - 0.3, bi["petal_llarg"].max() + 0.3, 100)
x2_vals = -(w1_final * x1_vals + b_final) / w2_final
plt.plot(x1_vals, x2_vals, "k--", linewidth=2, label="Frontera de scikit-learn")

plt.xlabel("Llargada del pètal (cm)")
plt.ylabel("Amplada del pètal (cm)")
plt.title("La frontera que ha après el model")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

La recta que dibuixa `LogisticRegression` no és cap misteri: és el punt vermell de la
secció anterior, el fons de la vall que hem trobat provant graella per graella. Nosaltres
hem trigat una estona provant combinacions; scikit-learn hi arriba en una fracció de
segon amb descens de gradient.

## 5. De 2 classes a 3

Iris té tres espècies, no dues. La sigmoide, tal com l'hem vista, només dona *sí/no*
entre dues classes. Per a tres (o més) classes, scikit-learn fa servir l'estratègia
**un contra la resta** (*one-vs-rest*): entrena **tres** models binaris per sota
(«és setosa o no?», «és versicolor o no?», «és virginica o no?») i, per a cada flor,
es queda amb l'espècie que hagi donat la probabilitat més alta.

Ho fem amb les mateixes dues columnes del pètal, ara amb les tres espècies.

In [ ]:
X3 = dades[["petal_llarg", "petal_ample"]].values
y3 = dades["especie"].values

X3tr, X3te, y3tr, y3te = train_test_split(X3, y3, test_size=0.3, random_state=42, stratify=y3)

model3 = LogisticRegression(max_iter=200, random_state=42)
model3.fit(X3tr, y3tr)

print(f"Precisió amb tres espècies: {model3.score(X3te, y3te):.1%} "
      f"({int(round(model3.score(X3te, y3te) * len(X3te)))} de {len(X3te)})")

exemples = dades.sample(4, random_state=42)
probs = model3.predict_proba(exemples[["petal_llarg", "petal_ample"]].values)

taula = pd.DataFrame(probs, columns=model3.classes_, index=exemples["especie"].values)
taula.index.name = "especie real"
taula["suma"] = taula.sum(axis=1)
taula.round(3)

Mireu la columna `suma`: sempre dona 1,000. Cada fila és una distribució de
probabilitat sencera repartida entre les tres espècies, no tres números independents.
Quan les tres probabilitats estan repartides (cap arriba a prop d'1), és senyal que la
flor és difícil de classificar; quan una s'enduu gairebé tot el pes, el model n'està
molt segur.

## 6. Pràctica

### Exercici 1 — La sigmoide, tu sol

Implementa la funció sigmoide (pots reaprofitar la idea de la secció 2, però escriu-la
de nou) i dibuixa-la per a $z$ entre $-8$ i $8$. Marca amb un punt on val exactament
0,5.

In [ ]:
# Exercici 1
# 1. Defineix la teva pròpia funció sigmoide (no reaprofitis la de dalt, escriu-la de nou).
# 2. Genera valors de z entre -8 i 8 amb np.linspace.
# 3. Dibuixa la corba amb plt.plot.
# 4. Marca amb plt.scatter el punt (0, 0.5).

### Exercici 2 — Calcula la probabilitat a mà

Agafa la flor amb pètal 5,0 × 1,5 cm. Fent servir `model.coef_` i `model.intercept_`
(el model binari versicolor/virginica), calcula tu mateix $z$ i $\sigma(z)$ amb
`numpy`. Després comprova que el resultat coincideix amb
`model.predict_proba(...)` per a aquesta mateixa flor.

In [ ]:
# Exercici 2
# 1. w1, w2 = model.coef_[0]; b = model.intercept_[0]
# 2. Calcula z = w1 * 5.0 + w2 * 1.5 + b
# 3. Calcula sigmoide(z) a mà
# 4. Compara amb model.predict_proba([[5.0, 1.5]]) — han de coincidir

### Exercici 3 — El paràmetre `C`

`LogisticRegression` té un paràmetre `C` que controla la regularització (l'ajust que
hem comentat a la secció 3, el que evita que els pesos creixin sense límit). Com més
petit és `C`, més es frena el model.

Entrena el model binari amb `C=0.01`, `C=1` i `C=100`, i dibuixa les tres fronteres de
decisió (com al codi de la secció 4) sobre el mateix núvol de punts. Què li passa a la
recta quan `C` és molt petit?

In [ ]:
# Exercici 3
# Per a cada valor de C a [0.01, 1, 100]:
#   1. Entrena LogisticRegression(C=valor, random_state=42) amb Xtr, ytr
#   2. Dibuixa la seva frontera (mateixa fórmula que a la secció 4)
#   3. Compara les tres rectes en un sol gràfic

### Exercici 4 — Altres columnes

Repeteix l'entrenament del model binari (versicolor/virginica) però amb
`sepal_llarg` i `sepal_ample` en lloc de les columnes del pètal. Mira la precisió a
l'examen i el núvol de punts.

Surt pitjor? Per què creus que les columnes del sèpal separen pitjor que les del
pètal? Mira els dos gràfics de dispersió costat a costat si t'ajuda a decidir-ho.

In [ ]:
# Exercici 4
# 1. X_sepal = bi[["sepal_llarg", "sepal_ample"]].values
# 2. Divideix en train/test igual que abans (mateix random_state)
# 3. Entrena un LogisticRegression nou i mira .score()
# 4. Dibuixa el núvol de punts amb aquestes dues columnes: se separen tan bé com abans?

## Resum

- La regressió logística tria una **recta** (o un pla, amb més columnes) i mesura de
  quin costat cau cada punt.
- La **sigmoide** converteix aquesta mesura en una **probabilitat**, entre 0 i 1.
- Els pesos s'aprenen fent petita una **funció de pèrdua** (log-loss): un termòmetre
  que castiga estar segur i equivocar-se. Ho hem vist com un **paisatge** amb un fons
  de vall, i baixar-hi és el que fa realment l'entrenament: el descens de gradient.
- Amb l'estratègia **un contra la resta**, la mateixa idea serveix per a més de dues
  classes.

### I ara què?

Una regressió logística no troba *la* recta que separa dues classes: troba **una**
recta, la que fa mínima la pèrdua. Però si les dades es poden separar bé, sovint hi ha
**moltes rectes** que ho fan raonablement. Quina és la millor?

Aquesta pregunta és exactament el punt de partida de l'**SVM** (*Support Vector
Machine*): en lloc de conformar-se amb qualsevol recta que separi, busca la que deixa
**el màxim marge** possible a banda i banda. Hi arribarem al proper quadern.